# L'impensable : le pétrole à prix négatif (avril 2020) · *The unthinkable: negative oil prices (April 2020)*

Notebook compagnon du chapitre **8. La carte du voyage : de la COVID au soft landing (2020-2025)** — [lire l'article](https://nmlab.io/ressources/de-la-covid-au-soft-landing-2020-2025).
Companion notebook to chapter **8. The Map of the Journey: From COVID to the Soft Landing (2020-2025)** — [read the article](https://nmlab.io/en/ressources/from-covid-to-the-soft-landing-2020-2025).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_oil() -> Series:
    """Prix quotidien du pétrole WTI spot (DCOILWTICO), 2019-2020, en direct de FRED —
    la série contient le −36,98 $ du 20 avril 2020.
    Daily WTI spot crude (DCOILWTICO), 2019-2020, live from FRED (incl. the negative print)."""
    return nm.load_fred("DCOILWTICO", start="2019").loc["2019":"2020"].dropna()

oil = load_oil()


from pandas import Timestamp as T
import matplotlib.dates as mdates
from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="L'impensable : le pétrole à prix négatif (avril 2020)",
        pre="~60 $ avant la pandémie",
        neg="20 avr. 2020 : −37 $",
        note="Source : EIA via FRED (DCOILWTICO), pétrole brut WTI spot (Cushing), quotidien, 2019-2020.\n"
             "20 avril 2020 : spot −36,98 $, contrat de mai réglé −37,63 $ — premier prix négatif de l'histoire du contrat."),
    "en": dict(
        title="The unthinkable: negative oil prices (April 2020)",
        pre="~$60 before the pandemic",
        neg="Apr. 20, 2020: −$37",
        note="Source: EIA via FRED (DCOILWTICO), WTI spot crude (Cushing), daily, 2019-2020.\n"
             "April 20, 2020: spot −$36.98, May contract settled at −$37.63 — the first negative price in the contract's history."),
}

def build_figure(oil: Series, lang: str) -> Figure:
    """Cours du pétrole, ligne de zéro, plongeon du 20 avril cerclé de rose."""
    text = LABELS[lang]
    import matplotlib
    matplotlib.rcParams["text.parse_math"] = False   # les « $ » restent littéraux
    def usd(v, _):
        s = "$" if lang == "en" else " $"
        if v < 0:
            return f"−${abs(v):.0f}" if lang == "en" else f"−{abs(v):.0f} $"
        return f"${v:.0f}" if lang == "en" else f"{v:.0f} $"
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig, left=0.078, top=0.86)
    ax.axhline(0, color=nm.COLORS["muted"], linestyle=(0, (6, 4)), linewidth=1.8, alpha=0.85)
    ax.plot(oil.index, oil, color=nm.COLORS["blue"], linewidth=2.0)

    trough = oil.loc["2020-04"].idxmin()
    tval = oil.loc[trough]
    ax.scatter([trough], [tval], s=150, facecolors="none", edgecolors=nm.COLORS["rose"],
               linewidths=2.6, zorder=6)

    ax.set_ylim(-46, 84)
    ax.set_yticks(range(-40, 81, 20))
    ax.yaxis.set_major_formatter(FuncFormatter(usd))
    ax.set_xlim(T("2019-01-01"), T("2021-01-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.annotate(text["pre"], xy=(T("2020-02-01"), 60), xytext=(T("2019-02-15"), 71),
                fontsize=22, color=nm.COLORS["text"], va="center", ha="left",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["neg"], xy=(trough, tval), xytext=(T("2020-06-01"), -29),
                fontsize=22, color=nm.COLORS["text"], va="center", ha="left",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    nm.header(fig, text["title"])
    nm.footer(fig, text["note"])
    return fig

build_figure(oil, LANG)